# Simple CUDA C++ Program on NVIDIA GPU (Google Colab)

This notebook writes, compiles, and runs a simple CUDA C++ program that performs **vector addition** on the GPU.

**Before running:** In Colab, go to `Runtime > Change runtime type` and set **Hardware accelerator** to **GPU** (T4 or better).

The program is generated as a `.cu` file, compiled with `nvcc` (already available on Colab GPU runtimes), and then executed.

In [1]:
# Verify a GPU is available and check the CUDA compiler version
!nvidia-smi
!nvcc --version

Sat Sep 19 09:04:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Write the CUDA C++ source file

The kernel `vectorAdd` adds two arrays element-wise on the GPU.

In [2]:
%%writefile vector_add.cu
#include <iostream>
#include <vector>
#include <cuda_runtime.h>

// CUDA kernel: element-wise addition of two vectors
__global__ void vectorAdd(const float* A, const float* B, float* C, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        C[idx] = A[idx] + B[idx];
    }
}

int main() {
    const int N = 1 << 20; // ~1 million elements
    size_t size = N * sizeof(float);

    // Host memory
    std::vector<float> h_A(N), h_B(N), h_C(N);
    for (int i = 0; i < N; ++i) {
        h_A[i] = static_cast<float>(i);
        h_B[i] = static_cast<float>(2 * i);
    }

    // Device memory
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    cudaMemcpy(d_A, h_A.data(), size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B.data(), size, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;
    vectorAdd<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N);
    cudaDeviceSynchronize();

    cudaMemcpy(h_C.data(), d_C, size, cudaMemcpyDeviceToHost);

    // Verify results
    bool success = true;
    for (int i = 0; i < N; ++i) {
        if (h_C[i] != h_A[i] + h_B[i]) {
            success = false;
            break;
        }
    }

    std::cout << (success ? "Vector addition successful!" : "Vector addition FAILED!") << std::endl;
    std::cout << "Sample: C[0]=" << h_C[0] << ", C[100]=" << h_C[100]
              << ", C[N-1]=" << h_C[N - 1] << std::endl;

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing vector_add.cu


## Compile the CUDA program with `nvcc`

In [3]:
!nvcc -o vector_add vector_add.cu

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


## Run the compiled program on the GPU

In [4]:
!./vector_add

Vector addition successful!
Sample: C[0]=0, C[100]=300, C[N-1]=3.14572e+06
